# Quora Question Pairs with DistilBERT

Binary paraphrase / duplicate detection: given `question1` and `question2`, predict `is_duplicate`.

**Exercise checklist**
- Pretrained model? → DistilBERT (+ a no-training Sentence-BERT baseline)
- Extra training? → fine-tune as a sentence-pair classifier
- How far without training? → cosine similarity on embeddings + threshold
- Phrasing sensitivity? → rewrite one question and re-score
- Label quality? → audit confident model–label disagreements

Data: [Kaggle Quora Question Pairs](https://www.kaggle.com/competitions/quora-question-pairs/overview)

## 0. Setup

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import gc
import torch
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, log_loss, roc_auc_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device={DEVICE}, torch={torch.__version__}")


def clear_memory():
    """Run GC and free MPS/CUDA cached memory. Clear caller refs yourself first (e.g. x = None)."""
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 96          # Quora questions are short; 96 is usually enough and faster
TRAIN_SAMPLE = 20_000    # set None for full train
EVAL_SAMPLE = 2_000      # smaller eval = much faster epoch end
NUM_EPOCHS = 3           # early stopping will usually cut this short
BATCH_SIZE = 48          # raise if memory allows; lower to 32 if MPS OOM
LR = 2e-5
NUM_PROC = max(1, (os.cpu_count() or 2) - 2)
EARLY_STOPPING_PATIENCE = 1


device=mps, torch=2.12.1


## 1. Load data

In [ ]:
KAGGLE_TRAIN_CSV = "train.csv"

if KAGGLE_TRAIN_CSV and os.path.exists(KAGGLE_TRAIN_CSV):
    raw = pd.read_csv(KAGGLE_TRAIN_CSV)
    raw = raw.dropna(subset=["question1", "question2", "is_duplicate"]).reset_index(drop=True)
    raw["label"] = raw["is_duplicate"].astype(int)

    from sklearn.model_selection import train_test_split

    train_df, val_df = train_test_split(
        raw, test_size=0.1, random_state=SEED, stratify=raw["label"]
    )
    ds = DatasetDict({
        "train": Dataset.from_pandas(train_df[["question1", "question2", "label"]], preserve_index=False),
        "validation": Dataset.from_pandas(val_df[["question1", "question2", "label"]], preserve_index=False),
    })
    print(f"Loaded Kaggle CSV: {KAGGLE_TRAIN_CSV}")

print(ds)
print(ds["train"][0])
print("label distribution (train):", pd.Series(ds["train"]["label"]).value_counts(normalize=True).to_dict())

Loaded Kaggle CSV: train.csv
DatasetDict({
    train: Dataset({
        features: ['question1', 'question2', 'label'],
        num_rows: 363858
    })
    validation: Dataset({
        features: ['question1', 'question2', 'label'],
        num_rows: 40429
    })
})
{'question1': 'Why is New York City the greatest city in the world?', 'question2': 'Why is New York one of the greatest cities in the world?', 'label': 1}
label distribution (train): {0: 0.6307982784492852, 1: 0.36920172155071485}


In [4]:
def maybe_sample(split, n):
    if n is None or n >= len(split):
        return split
    return split.shuffle(seed=SEED).select(range(n))

train_ds = maybe_sample(ds["train"], TRAIN_SAMPLE)
eval_split_name = "validation" if "validation" in ds else "validation_matched"
if eval_split_name not in ds:
    eval_split_name = list(ds.keys())[1]
eval_ds = maybe_sample(ds[eval_split_name], EVAL_SAMPLE)

print(f"train={len(train_ds):,}  eval={len(eval_ds):,}  (split={eval_split_name})")

train=20,000  eval=2,000  (split=validation)


## 2. Quick label sanity checks (before any model)

Cheap heuristics that often catch noisy or suspicious labels.

In [5]:
def normalize(text: str) -> str:
    return " ".join(str(text).lower().split())

probe = ds["train"].shuffle(seed=SEED).select(range(min(50_000, len(ds["train"]))))
rows = []
for ex in probe:
    q1, q2, y = ex["question1"], ex["question2"], ex["label"]
    n1, n2 = normalize(q1), normalize(q2)
    rows.append({
        "question1": q1,
        "question2": q2,
        "label": y,
        "exact_match": int(n1 == n2),
        "jaccard": len(set(n1.split()) & set(n2.split())) / max(1, len(set(n1.split()) | set(n2.split()))),
    })

probe_df = pd.DataFrame(rows)
suspicious_exact = probe_df[(probe_df["exact_match"] == 1) & (probe_df["label"] == 0)]
suspicious_low_overlap_dup = probe_df[(probe_df["jaccard"] < 0.15) & (probe_df["label"] == 1)].head(10)
suspicious_high_overlap_non = probe_df[(probe_df["jaccard"] > 0.85) & (probe_df["label"] == 0)].head(10)

print(f"Exact-match pairs labeled NOT duplicate: {len(suspicious_exact)}")
display(suspicious_exact.head(5))
print("High lexical overlap but label=0 (sample):")
display(suspicious_high_overlap_non)
print("Low lexical overlap but label=1 (sample):")
display(suspicious_low_overlap_dup)

Exact-match pairs labeled NOT duplicate: 0


,question1,question2,label,exact_match,jaccard


High lexical overlap but label=0 (sample):


,question1,question2,label,exact_match,jaccard
12,What is it like flying from Bangkok to Manila ...,What is it like flying from Bangkok to Manila ...,0,0,0.888889
51,What are the strongest majors in terms of job ...,What are the strongest majors in terms of job ...,0,0,0.875000
149,"What's classy if you’re rich, but trashy if yo...","What's trashy if you're rich, but classy if yo...",0,0,0.888889
155,What are some things new employees should know...,What are some things new employees should know...,0,0,0.882353
376,What is the importance of education to the Uni...,What is the importance of education to the Uni...,0,0,0.888889
443,What are the bitter truths of going to Germany...,What are the bitter truths of going to austral...,0,0,0.866667
556,How can I use Omegle on my iPod Touch and how ...,How can I use Omegle on my iPod Touch and how ...,0,0,0.882353
561,Who are some lesser known important historical...,Who are some lesser known important historical...,0,0,0.888889
579,If a Bollywood movie were made on the life and...,If a Bollywood movie were made on the life and...,0,0,0.863636
587,What individuals and events in history are a s...,What individuals and events in history are a s...,0,0,0.857143


Low lexical overlap but label=1 (sample):


,question1,question2,label,exact_match,jaccard
22,Why is Hillary Clinton considered corrupt?,Are the Clintons corrupt?,1,0,0.111111
118,Can we control our feelings?,How do you control your emotions in public?,1,0,0.083333
119,Why do some people call Mustafa Kemal Ataturk ...,Was Kemal Ataturk a democrat theoritically but...,1,0,0.117647
368,How can I stop my porn addiction?,What is the best way to overcome an porn addic...,1,0,0.133333
381,"Why do we humans exist, what is our purpose in...",How can I find myself or purpose of life?,1,0,0.111111
405,How does it feel to be a Christian in Pakistan?,What is the situation like for Christians in P...,1,0,0.117647
423,Is the surgical strike against Pakistan fake?,Did India really conduct a surgical strike on ...,1,0,0.142857
532,What are the minimum requirements to enter MIT?,How can I be selected in mit?,1,0,0.071429
609,What is the best way to avoid procrastination?,How do I stop procrastination?,1,0,0.083333
658,Does long distance relationship work?,How do we can remain satisfied in a long dista...,1,0,0.142857


## 3. How far without training?

Use a pretrained SentenceTransformer (transfer learning, **no fine-tuning**): embed both questions, take cosine similarity, pick a threshold on the eval split.

In [ ]:
from sentence_transformers import SentenceTransformer

clear_memory()

# Keep ST on MPS but encode to NumPy in small batches so we never hold two huge GPU tensors.
ST_BATCH = 64 if DEVICE == "mps" else 256
st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

def pair_cosine_scores_fast(dataset, model, batch_size=ST_BATCH):
    """Pairwise cosine for aligned (q1_i, q2_i). Returns NumPy; frees GPU between passes."""
    e1 = model.encode(
        dataset["question1"],
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    clear_memory()
    e2 = model.encode(
        dataset["question2"],
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    scores = (e1 * e2).sum(axis=-1).astype(np.float32)
    del e1, e2
    clear_memory()
    return scores

base_scores = pair_cosine_scores_fast(eval_ds, st_model)
base_labels = np.array(eval_ds["label"])

thresholds = np.linspace(0.2, 0.9, 29)
rows = []
for t in thresholds:
    pred = (base_scores >= t).astype(int)
    rows.append({
        "threshold": t,
        "accuracy": accuracy_score(base_labels, pred),
        "f1": f1_score(base_labels, pred),
    })
tune_df = pd.DataFrame(rows)
best = tune_df.loc[tune_df["f1"].idxmax()]
best_t = float(best["threshold"])
base_pred = (base_scores >= best_t).astype(int)
base_prob = np.clip(
    (base_scores - base_scores.min()) / (base_scores.max() - base_scores.min() + 1e-8),
    1e-6, 1 - 1e-6,
)

print(f"Best threshold={best_t:.3f}")
print(
    f"No-training baseline — acc={accuracy_score(base_labels, base_pred):.4f}  "
    f"f1={f1_score(base_labels, base_pred):.4f}  "
    f"auc={roc_auc_score(base_labels, base_scores):.4f}  "
    f"logloss={log_loss(base_labels, base_prob):.4f}"
)

# Free the SentenceTransformer before DistilBERT fine-tuning
st_model = None
clear_memory()
print("Cleared SentenceTransformer from memory.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Best threshold=0.725
No-training baseline — acc=0.7545  f1=0.7221  auc=0.8653  logloss=0.8136
Cleared SentenceTransformer from memory.


### Phrasing sensitivity (still no training)

Slight rewrites should keep high similarity if the model captures meaning, not surface form.

In [ ]:
_st_cpu = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

phrasing_pairs = [
    (
        "How can I learn Python programming effectively?",
        "What's the best way to get good at coding in Python?",
        "expected: high similarity (paraphrase)",
    ),
    (
        "How can I learn Python programming effectively?",
        "How can I learn Python programming effectively?",
        "expected: ~1.0 (identical)",
    ),
    (
        "How can I learn Python programming effectively?",
        "How can I learn Java programming effectively?",
        "expected: medium (same template, different topic)",
    ),
    (
        "What is the capital of France?",
        "How do I bake sourdough bread at home?",
        "expected: low similarity",
    ),
]

for q1, q2, note in phrasing_pairs:
    e1 = _st_cpu.encode(q1, convert_to_numpy=True, normalize_embeddings=True)
    e2 = _st_cpu.encode(q2, convert_to_numpy=True, normalize_embeddings=True)
    sim = float((e1 * e2).sum())
    print(f"{sim:.3f} | {note}")
    print(f"  Q1: {q1}")
    print(f"  Q2: {q2}\n")

_st_cpu = None
clear_memory()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.818 | expected: high similarity (paraphrase)
  Q1: How can I learn Python programming effectively?
  Q2: What's the best way to get good at coding in Python?

1.000 | expected: ~1.0 (identical)
  Q1: How can I learn Python programming effectively?
  Q2: How can I learn Python programming effectively?

0.578 | expected: medium (same template, different topic)
  Q1: How can I learn Python programming effectively?
  Q2: How can I learn Java programming effectively?

0.100 | expected: low similarity
  Q1: What is the capital of France?
  Q2: How do I bake sourdough bread at home?



## 4. Fine-tune DistilBERT as a sentence-pair classifier

Input format: `[CLS] question1 [SEP] question2 [SEP]` → linear head on pooled output → `P(duplicate)`.

This is **fine-tuning** (update DistilBERT weights), not frozen feature extraction.

In [ ]:
clear_memory()

from transformers import EarlyStoppingCallback

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["question1"],
        batch["question2"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

_tokenize_procs = 1 if DEVICE == "mps" else NUM_PROC
print(f"Tokenizing train={len(train_ds):,} eval={len(eval_ds):,} (num_proc={_tokenize_procs})...")

def _map_tok(dataset, desc):
    kwargs = dict(
        function=tokenize_fn,
        batched=True,
        batch_size=1024,
        remove_columns=[c for c in dataset.column_names if c not in ("label",)],
        desc=desc,
    )
    if _tokenize_procs > 1:
        try:
            return dataset.map(**kwargs, num_proc=_tokenize_procs)
        except Exception as e:
            print(f"Parallel tokenize failed ({e}); falling back to num_proc=1")
    return dataset.map(**kwargs)

tokenized_train = _map_tok(train_ds, "tokenize train")
tokenized_eval = _map_tok(eval_ds, "tokenize eval")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "roc_auc": roc_auc_score(labels, probs),
        "log_loss": log_loss(labels, np.clip(probs, 1e-6, 1 - 1e-6)),
    }

output_dir = "./distilbert-qqp"
args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    num_train_epochs=NUM_EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    optim="adamw_torch",
    warmup_ratio=0.06,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print(
    f"Training on {DEVICE} | epochs≤{NUM_EPOCHS} | batch={BATCH_SIZE} | "
    f"train={len(tokenized_train):,} | eval={len(tokenized_eval):,}"
)
train_result = trainer.train()
print(train_result)
eval_result = trainer.evaluate()
print("Fine-tuned DistilBERT:", eval_result)
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")


Tokenizing train=20,000 eval=2,000 (num_proc=1)...


tokenize train:   0%|          | 0/20000 [00:00<?, ? examples/s]

tokenize eval:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training on mps | epochs≤3 | batch=48 | train=20,000 | eval=2,000


... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1251, training_loss=0.3694179147648678, metrics={'train_runtime': 746.5337, 'train_samples_per_second': 80.371, 'train_steps_per_second': 1.676, 'total_flos': 1101027621566592.0, 'train_loss': 0.3694179147648678, 'epoch': 3.0})


Training Loss,Validation Loss,Epoch,Accuracy,F1,Roc Auc,Log Loss,Runtime,Samples Per Second,Steps Per Second
0.278724,0.395437,3,0.826000,0.769841,0.907277,0.395437,5.524900,362.000000,3.801000


Fine-tuned DistilBERT: {'eval_loss': 0.395436555147171, 'eval_accuracy': 0.826, 'eval_f1': 0.7698412698412699, 'eval_roc_auc': 0.9072768098191828, 'eval_log_loss': 0.39543660124475327, 'eval_runtime': 5.5249, 'eval_samples_per_second': 362.0, 'eval_steps_per_second': 3.801}
Best checkpoint: ./distilbert-qqp/checkpoint-1251


## 5. Error analysis & label audit

Look at high-confidence mistakes — often label noise, sometimes real model failures.

In [9]:
pred_out = trainer.predict(tokenized_eval)
logits = pred_out.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
preds = np.argmax(logits, axis=-1)
labels = np.array(eval_ds["label"])

audit = pd.DataFrame({
    "question1": eval_ds["question1"],
    "question2": eval_ds["question2"],
    "label": labels,
    "pred": preds,
    "prob_duplicate": probs,
})
audit["correct"] = audit["label"] == audit["pred"]
audit["confidence"] = np.maximum(audit["prob_duplicate"], 1 - audit["prob_duplicate"])

false_pos = audit[(audit["label"] == 0) & (audit["pred"] == 1)].sort_values("prob_duplicate", ascending=False)
false_neg = audit[(audit["label"] == 1) & (audit["pred"] == 0)].sort_values("prob_duplicate", ascending=True)

print(f"Errors: FP={len(false_pos):,}  FN={len(false_neg):,}  / {len(audit):,}")
print("\nTop confident FALSE POSITIVES (model says duplicate, label=0) — check labels:")
display(false_pos.head(8)[["question1", "question2", "label", "prob_duplicate"]])
print("\nTop confident FALSE NEGATIVES (model says different, label=1) — check labels:")
display(false_neg.head(8)[["question1", "question2", "label", "prob_duplicate"]])

Errors: FP=228  FN=120  / 2,000

Top confident FALSE POSITIVES (model says duplicate, label=0) — check labels:


,question1,question2,label,prob_duplicate
42,Are there any ways to increase nose size witho...,Are there any ways to reduce nose size without...,0,0.968619
13,Would Donald Trump be soon assassinated?,What is the possibility of an assassination of...,0,0.961056
651,Which is the best Android smartphone under Rs ...,Which is the best android basic smartphone aro...,0,0.959266
1479,Who are some of the most deserving candidates ...,Who are some of the most deserving candidates ...,0,0.958433
1860,Are answers on Quora verified for their authen...,How can I believe in the answers given on Quor...,0,0.953805
1870,How can I find out what I want to do with my l...,How can I find out what I really want to do wi...,0,0.944370
993,What are the mistakes which you did while prep...,What are some of the mistakes you did while pr...,0,0.944231
366,What should I do if I am happy and satisfied w...,What should I do if I am not happy in my life?,0,0.943689



Top confident FALSE NEGATIVES (model says different, label=1) — check labels:


,question1,question2,label,prob_duplicate
1738,What are the best ways to spend the weekends f...,How do I spend the weekends in Bangalore if I ...,1,0.003553
300,Which hashtag will I use to get more likes in ...,What are the best hashtags to use as a photogr...,1,0.007662
1080,How do I cancel a Facebook invitation that I s...,"Can I reverse an invitation to ""like"" a Facebo...",1,0.007695
554,Who is the next MJ?,Who is going to be the next Michael Jackson?,1,0.009150
1340,What are the modern schools of thought in lega...,What are the different schools of thought with...,1,0.010692
1484,Why do Android OS updates have names based on ...,Why are android code names named after confect...,1,0.011659
1555,How can I use FaceTime on my laptop?,How can you get FaceTime for a Mac?,1,0.012690
1705,How do I write journal report?,What do you write about in your journal?,1,0.013470


## 6. Compare approaches

In [10]:
ft_preds = preds
ft_probs = np.clip(probs, 1e-6, 1 - 1e-6)

comparison = pd.DataFrame([
    {
        "approach": "No training (MiniLM cosine)",
        "accuracy": accuracy_score(base_labels, base_pred),
        "f1": f1_score(base_labels, base_pred),
        "roc_auc": roc_auc_score(base_labels, base_scores),
        "log_loss": log_loss(base_labels, base_prob),
    },
    {
        "approach": "Fine-tuned DistilBERT",
        "accuracy": accuracy_score(labels, ft_preds),
        "f1": f1_score(labels, ft_preds),
        "roc_auc": roc_auc_score(labels, ft_probs),
        "log_loss": log_loss(labels, ft_probs),
    },
])
display(comparison.round(4))

print("""
Takeaways for the exercise:
- Yes, pretrained models help a lot (MiniLM / DistilBERT).
- Extra training (fine-tuning DistilBERT on pairs) usually beats frozen embeddings.
- Without training you can still get a decent baseline via cosine similarity.
- Phrasing changes matter: paraphrases stay high; topic swaps drop; identical ≈ 1.
- Label noise is real — inspect confident disagreements before trusting every gold label.
""")

,approach,accuracy,f1,roc_auc,log_loss
0,No training (MiniLM cosine),0.7545,0.7221,0.8653,0.8136
1,Fine-tuned DistilBERT,0.8260,0.7698,0.9073,0.3954



Takeaways for the exercise:
- Yes, pretrained models help a lot (MiniLM / DistilBERT).
- Extra training (fine-tuning DistilBERT on pairs) usually beats frozen embeddings.
- Without training you can still get a decent baseline via cosine similarity.
- Phrasing changes matter: paraphrases stay high; topic swaps drop; identical ≈ 1.
- Label noise is real — inspect confident disagreements before trusting every gold label.



## 7. Summary

### Problem
Quora Question Pairs is **binary paraphrase / duplicate detection**: given two questions, predict whether they ask the same thing (`is_duplicate`).

### Answers to the exercise questions

| Question | Answer in this notebook |
|----------|-------------------------|
| Can you use a pretrained model? | **Yes** — MiniLM (no-training baseline) and DistilBERT (fine-tuned classifier). |
| Do you need additional training? | **For a strong score, yes.** Fine-tuning DistilBERT as a sentence-pair classifier usually beats frozen embeddings. Transfer learning without updates is still a solid baseline. |
| How far without training? | Cosine similarity on SentenceTransformer embeddings + a threshold already gives a usable baseline. Phrasing changes (paraphrase vs topic swap vs unrelated) move the score in the expected direction. |
| Can you test that labels are correct? | **Partly.** Lexical heuristics (exact match / Jaccard) and high-confidence model–label disagreements surface suspicious pairs; Quora labels are known to be noisy. |


### Results (this run)

Obtained with `TRAIN_SAMPLE = 20_000` and `EVAL_SAMPLE = 2_000`.


| approach | accuracy | f1 | roc_auc | log_loss |
|----------|----------|-----|---------|----------|
| No training (MiniLM cosine) | 0.7545 | 0.7221 | 0.8653 | 0.8136 |
| Fine-tuned DistilBERT | **0.8260** | **0.7698** | **0.9073** | **0.3954** |

Fine-tuned DistilBERT wins on every metric. Biggest gains: **log loss** (0.81 → 0.40) and **accuracy** (+7.2 pts), which matches the idea that task-specific fine-tuning beats frozen embeddings for calibrated duplicate probabilities.

### Pipeline we used
1. Load Kaggle `train.csv` (or HF GLUE QQP)
2. Label sanity checks
3. No-training MiniLM cosine baseline + phrasing probes
4. Fine-tune DistilBERT with early stopping on validation F1
5. Error analysis / label audit
6. Optional batched `test.csv` → `submission.csv`

### Next steps for a stronger score
- `TRAIN_SAMPLE = None`, `EVAL_SAMPLE = None`, 2–3 epochs with early stopping
- Try `roberta-base` or a QQP-tuned checkpoint
- Soft-label or drop obvious label noise
- Ensemble a few seeds
